# Лабораторна робота № 3
## Діагностика шкал вимірювання

Курс СК26 «Інтелектуальний аналіз та візуалізація даних»

**Тривалість:** 2 год аудиторно + 3 год самостійно · **Оцінка:** 4 бали
**Файли:** `Ковід_дані_навчальні.xlsx`, `ЛР3_помилковий_аналіз.ipynb`

---

Шкала вимірювання визначає, які операції над змінною взагалі мають сенс.
Комп'ютер цього не знає: він порахує середнє від номера телефону й не поскаржиться.
Знати має вимірювач.

У цій роботі ви складаєте **паспорт восьми змінних** свого варіанта, показуєте на
власних даних, які статистики для них недопустимі, і виправляєте чужий аналіз,
де таких помилок десять.

In [1]:
ПРІЗВИЩЕ_ІМЯ = 'Токарик Сергій'      # напишіть тут своє прізвище та ім'я
ГРУПА        = 'ШІ-33'
НОМЕР        = 7       # ваш номер за списком групи (1–30)

assert ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР, 'Заповніть три поля вище'
assert 1 <= НОМЕР <= 30, 'Номер має бути від 1 до 30'

In [2]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_colwidth', 60)

ВАРІАНТИ = json.load(open('lr3_variants_student.json', encoding='utf-8'))
В = ВАРІАНТИ[НОМЕР - 1]

print('=' * 74)
print(f'ВАРІАНТ {В["варіант"]}   ·   {ПРІЗВИЩЕ_ІМЯ}   ·   {ГРУПА}')
print('=' * 74)
print()
print('Ваші вісім стовпців:')
for i, c in enumerate(В['стовпці'], 1):
    print(f'   {i}. {c}')
print()
print('Ваша пара для аналізу зв’язку:', ' та '.join(В['пара']))
print()
print('Ваше додаткове питання:')
print('  ', В['питання'])

ВАРІАНТ 7   ·   Токарик Сергій   ·   ШІ-33

Ваші вісім стовпців:
   1. ПІП
   2. помер/жив
   3. ступінь КТ
   4. Дата розтину
   5. КТ
   6. Т вище 37,5
   7. Сечов1
   8. БРпр1

Ваша пара для аналізу зв’язку: Стать та помер/жив

Ваше додаткове питання:
   Ви хочете нормалізувати всі числові стовпці формулою (x − min) / (max − min). Знайдіть у файлі стовпець, на якому ця формула зламається, і поясніть чому.


In [3]:
ФАЙЛ = 'Ковід_дані_навчальні.xlsx'
df = pd.read_excel(ФАЙЛ, sheet_name='Лист1', engine='openpyxl')
МОЇ = В['стовпці']

print('Розмір набору:', df.shape)
df[МОЇ].head(8)

Розмір набору: (121, 77)


,ПІП,помер/жив,ступінь КТ,Дата розтину,КТ,"Т вище 37,5",Сечов1,БРпр1
0,Федорюк Петро Юрійович,1,сердній,2022-03-11,20-30%,3,5.1,3.4
1,Загул Григорій Сергійович,2,середній,NaT,30-35%,3,6.6,2.8
2,Бондара Лариса Павлівна,2,критичний,NaT,70-80%,3,7.0,2.8
3,Ватаманюка Уляна Дмитрівна,2,критичний,NaT,0.8,6,9.5,3.4
4,Бабійа Лариса Олегівна,2,NaN,NaT,NaN,5 до гос,5.2,3.4
5,Ковалика ганна Андріївна,1,середній,2021-12-26,30-40%,NaN,14.0,3.4
6,Височан Віктор Володимирович,2,важкий,NaT,60-65%,NaN,5.7,5.1
7,Дзюбаа Ірина Іванівна,2,легкий,NaT,до 17%,NaN,9.1,5.1


---
# Етап 1. Паспорт змінних

Для кожного з восьми стовпців визначте шкалу вимірювання. Спирайтеся не на тип даних
у pandas, а на **зміст величини**: що означають її значення і які порівняння між ними
мають сенс.

Нагадування:

| Шкала | Ознака |
|---|---|
| Номінальна | значення можна лише порівняти на рівність |
| Порядкова | є природний порядок, але відстані між сусідніми рівнями невідомі |
| Інтервальна | різниці мають сенс, нуль умовний |
| Відношень | є природний нуль, відношення мають сенс |
| Абсолютна | природні і нуль, і одиниця вимірювання |

Окремо позначайте **ідентифікатори** — значення, що не є величиною взагалі
(номер історії хвороби, прізвище, телефон).

In [4]:
# Спершу подивіться, що каже pandas — і не довіряйте йому
df[МОЇ].info()

<class 'pandas.DataFrame'>
RangeIndex: 121 entries, 0 to 120
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   ПІП           121 non-null    str           
 1   помер/жив     121 non-null    int64         
 2   ступінь КТ    42 non-null     str           
 3   Дата розтину  61 non-null     datetime64[us]
 4   КТ            44 non-null     object        
 5   Т вище 37,5   78 non-null     object        
 6   Сечов1        117 non-null    float64       
 7   БРпр1         109 non-null    float64       
dtypes: datetime64[us](1), float64(2), int64(1), object(2), str(2)
memory usage: 7.7+ KB


In [5]:
# Далі — що є всередині насправді
for c in МОЇ:
    з = df[c].dropna()
    типи = ', '.join(f'{t.__name__}×{n}' for t, n in з.map(type).value_counts().items())
    print(f'--- {c}')
    print(f'    непорожніх {len(з)} з {len(df)} | унікальних {з.nunique()} | типи: {типи}')
    print(f'    приклади: {[repr(v) for v in з.unique()[:4]]}')
    print()

--- ПІП
    непорожніх 121 з 121 | унікальних 121 | типи: str×121
    приклади: ["'Федорюк Петро Юрійович'", "'Загул Григорій Сергійович'", "'Бондара Лариса Павлівна'", "'Ватаманюка Уляна Дмитрівна'"]

--- помер/жив
    непорожніх 121 з 121 | унікальних 2 | типи: int×121
    приклади: ['np.int64(1)', 'np.int64(2)']

--- ступінь КТ
    непорожніх 42 з 121 | унікальних 12 | типи: str×42
    приклади: ["'сердній'", "'середній'", "'критичний'", "'важкий'"]

--- Дата розтину
    непорожніх 61 з 121 | унікальних 50 | типи: Timestamp×61
    приклади: ["Timestamp('2022-03-11 00:00:00')", "Timestamp('2021-12-26 00:00:00')", "Timestamp('2021-11-19 00:00:00')", "Timestamp('2022-05-13 00:00:00')"]

--- КТ
    непорожніх 44 з 121 | унікальних 26 | типи: float×27, str×17
    приклади: ["'20-30%'", "'30-35%'", "'70-80%'", '0.8']

--- Т вище 37,5
    непорожніх 78 з 121 | унікальних 35 | типи: str×48, int×30
    приклади: ['3', '6', "'5 до гос'", "'3 дні в стац'"]

--- Сечов1
    непорожніх 117 з 121 

### Заповніть паспорт

Для кожного стовпця вкажіть шкалу та **обґрунтування** — одне речення про те,
чому саме ця шкала. Обґрунтування оцінюється, назва шкали сама по собі — ні.

In [6]:
ПАСПОРТ = {
    'ПІП': (
        'ідентифікатор',
        'Унікальне текстове ім\'я пацієнта слугує для його ідентифікації та не є вимірюваною числовою величиною.'
    ),
    'помер/жив': (
        'номінальна',
        'Категоріальний бінарний показник стану пацієнта, коди якого не мають математичного порядку чи числового значення.'
    ),
    'ступінь КТ': (
        'порядкова',
        'Категорії ураження легень мають чітку градацію за важкістю, але точні числові інтервали між ними не визначені.'
    ),
    'Дата розтину': (
        'інтервальна',
        'Календарна дата має довільний відліковий нуль, тому обчислювати можна лише різниці між датами (тривалість).'
    ),
    'КТ': (
        'відношень',
        'Відсоток ураження легень має природний нуль (0% ураження) та дозволяє порівнювати пропорційні відношення величин.'
    ),
    'Т вище 37,5': (
        'відношень',
        'Показник вимірює кількість днів тривалості лихоманки, де нуль означає повну відсутність днів із підвищеною температурою.'
    ),
    'Сечов1': (
        'відношень',
        'Концентрація речовини в крові є фізичною величиною з абсолютним нулем, де можливе порівнення у скільки разів.'
    ),
    'БРпр1': (
        'відношень',
        'Лабораторний показник вмісту білірубіну володіє природним нульовим рівнем та допускає обчислення відношень значень.'
    ),
}

паспорт_df = pd.DataFrame(
    [(c, ПАСПОРТ[c][0], ПАСПОРТ[c][1]) for c in МОЇ],
    columns=['Стовпець', 'Шкала', 'Обґрунтування']
)
паспорт_df

,Стовпець,Шкала,Обґрунтування
0,ПІП,ідентифікатор,Унікальне текстове ім'я пацієнта слугує для його ідентиф...
1,помер/жив,номінальна,"Категоріальний бінарний показник стану пацієнта, коди як..."
2,ступінь КТ,порядкова,Категорії ураження легень мають чітку градацію за важкіс...
3,Дата розтину,інтервальна,"Календарна дата має довільний відліковий нуль, тому обчи..."
4,КТ,відношень,Відсоток ураження легень має природний нуль (0% ураження...
5,"Т вище 37,5",відношень,"Показник вимірює кількість днів тривалості лихоманки, де..."
6,Сечов1,відношень,Концентрація речовини в крові є фізичною величиною з абс...
7,БРпр1,відношень,Лабораторний показник вмісту білірубіну володіє природни...


In [7]:
# --- САМОПЕРЕВІРКА 1 ---
ДОПУСТИМІ = {'номінальна', 'порядкова', 'інтервальна', 'відношень',
             'абсолютна', 'ідентифікатор'}
for c in МОЇ:
    ш, об = ПАСПОРТ[c]
    assert ш.strip().lower() in ДОПУСТИМІ, f'{c}: шкалу «{ш}» не розпізнано'
    assert len(об.strip()) >= 25, f'{c}: обґрунтування закоротке'
print('Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.')

Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.


---
# Етап 2. Допустимі статистики

Для кожного стовпця обчисліть **тільки те, що дозволяє його шкала**.

* номінальна та ідентифікатор — мода, частоти
* порядкова — додатково медіана та квантилі
* інтервальна — додатково середнє та стандартне відхилення
* відношень і абсолютна — усе, зокрема коефіцієнт варіації

Якщо для стовпця не можна порахувати нічого, крім частот, — так і напишіть.

In [8]:
def допустима_статистика(стовпець, шкала):

    з = df[стовпець].dropna()
    р = {'непорожніх': len(з), 'унікальних': з.nunique()}
    ш = шкала.strip().lower()

    if ш in ('ідентифікатор',):
        return р

    if not з.empty:
        р['мода'] = з.mode().iloc[0] if not з.mode().empty else None
        р['топ-частоти'] = dict(з.value_counts().head(3))

    if ш == 'номінальна':
        return р

    if pd.api.types.is_datetime64_any_dtype(з):
        з_num = (з - з.min()).dt.days.astype(float)
    else:
        з_num = pd.to_numeric(з, errors='coerce').dropna()

    if ш in ('порядкова', 'інтервальна', 'відношень', 'абсолютна'):
        if not з_num.empty:
            р['медіана'] = np.round(з_num.median(), 2)
            р['Q1 (25%)'] = np.round(з_num.quantile(0.25), 2)
            р['Q3 (75%)'] = np.round(з_num.quantile(0.75), 2)

    if ш == 'порядкова':
        return р

    if ш in ('інтервальна', 'відношень', 'абсолютна'):
        if not з_num.empty:
            р['середнє'] = np.round(з_num.mean(), 2)
            р['ст. відхилення'] = np.round(з_num.std(), 2)

    if ш == 'інтервальна':
        return р

    if ш in ('відношень', 'абсолютна'):
        if not з_num.empty and з_num.mean() != 0:
            cv = (з_num.std() / з_num.mean()) * 100
            р['коеф. варіації (%)'] = np.round(cv, 2)

    return р


for c in МОЇ:
    print(f'--- {c}  ({ПАСПОРТ[c][0]})')
    print('   ', допустима_статистика(c, ПАСПОРТ[c][0]))
    print()

--- ПІП  (ідентифікатор)
    {'непорожніх': 121, 'унікальних': 121}

--- помер/жив  (номінальна)
    {'непорожніх': 121, 'унікальних': 2, 'мода': np.int64(1), 'топ-частоти': {1: np.int64(61), 2: np.int64(60)}}

--- ступінь КТ  (порядкова)
    {'непорожніх': 42, 'унікальних': 12, 'мода': 'середній', 'топ-частоти': {'середній': np.int64(10), 'важкий': np.int64(9), 'серед ступ': np.int64(6)}}

--- Дата розтину  (інтервальна)
    {'непорожніх': 61, 'унікальних': 50, 'мода': Timestamp('2022-06-07 00:00:00'), 'топ-частоти': {Timestamp('2022-06-07 00:00:00'): np.int64(3), Timestamp('2022-04-07 00:00:00'): np.int64(2), Timestamp('2022-04-27 00:00:00'): np.int64(2)}, 'медіана': np.float64(472.0), 'Q1 (25%)': np.float64(447.0), 'Q3 (75%)': np.float64(486.0), 'середнє': np.float64(455.56), 'ст. відхилення': np.float64(72.57)}

--- КТ  (відношень)
    {'непорожніх': 44, 'унікальних': 26, 'мода': 0.3, 'топ-частоти': {0.3: np.int64(6), 0.25: np.int64(5), 0.8: np.int64(3)}, 'медіана': np.float64(0.3)

---
# Етап 3. Три безглузді числа

`df.describe()` порахує статистики для всього, що схоже на число. Частина цих чисел
не має жодного змісту.

Знайдіть **щонайменше три** такі числа у своєму наборі: обчисліть їх, а потім
поясніть, чому кожне з них безглузде.

In [9]:
df[МОЇ].describe(include='all').T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
ПІП,121,121,Федорюк Петро Юрійович,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
помер/жив,121.0,NaN,NaN,NaN,1.495868,1.0,1.0,1.0,2.0,2.0,0.502062
ступінь КТ,42,12,середній,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Дата розтину,61,NaN,NaN,NaN,2022-04-06 13:22:37.377049,2021-01-06 00:00:00,2022-03-29 00:00:00,2022-04-23 00:00:00,2022-05-07 00:00:00,2022-06-22 00:00:00,NaN
КТ,44.0,26.0,0.3,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
"Т вище 37,5",78.0,35.0,3.0,13.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Сечов1,117.0,NaN,NaN,NaN,8.93188,3.1,5.5,7.2,9.5,68.0,8.174368
БРпр1,109.0,NaN,NaN,NaN,4.801468,2.4,3.4,5.1,5.3,10.2,1.575191


In [10]:
дати = df['Дата розтину'].dropna()
дні = (дати - pd.Timestamp('1970-01-01')).dt.days
cv_дат = 100 * дні.std() / дні.mean()

БЕЗГЛУЗДІ = [
    (
        'помер/жив',
        'середнє (mean)',
        np.round(df['помер/жив'].mean(), 2),
        'Оскільки це номінальна бінарна змінна (коди 1 та 2), середнє арифметичне значень кодів не має жодного фізичного чи медичного змісту.'
    ),
    (
        'помер/жив',
        'стандартне відхилення (std)',
        np.round(df['помер/жив'].std(), 2),
        'Для номінальних даних відхилення від умовного «середнього коду» є математичним нонсенсом, бо між кодами немає чисельної відстані.'
    ),
    (
        'помер/жив',
        'медіана (50%)',
        np.round(df['помер/жив'].median(), 2),
        'Номінальна шкала не має впорядкованості, тому поняття центрального елемента в упорядкованому ряду для неї застосувати неможливо.'
    ),
    (
        'Дата розтину',
        'коефіцієнт варіації (%)',
        np.round(cv_дат, 3),
        'Дата належить до інтервальної шкали з умовним нулем (1970 рік обрано довільно): якщо змінити точку відліку, середнє зміниться, а отже зміниться й відношення std/mean.'
    ),
]

assert len(БЕЗГЛУЗДІ) >= 3, 'Потрібно щонайменше три приклади'
pd.DataFrame(БЕЗГЛУЗДІ, columns=['Стовпець', 'Статистика', 'Значення', 'Чому безглузде'])

,Стовпець,Статистика,Значення,Чому безглузде
0,помер/жив,середнє (mean),1.50,"Оскільки це номінальна бінарна змінна (коди 1 та 2), сер..."
1,помер/жив,стандартне відхилення (std),0.50,Для номінальних даних відхилення від умовного «середньог...
2,помер/жив,медіана (50%),1.00,"Номінальна шкала не має впорядкованості, тому поняття це..."
3,Дата розтину,коефіцієнт варіації (%),0.38,Дата належить до інтервальної шкали з умовним нулем (197...


---
# Етап 4. Міра зв'язку для вашої пари

Для двох змінних вашого варіанта треба оцінити зв'язок. Міру обирає **пара шкал**,
а не зручність:

| Поєднання шкал | Міра |
|---|---|
| відношень × відношень | кореляція Пірсона |
| порядкова × порядкова | рангова кореляція Спірмена |
| номінальна × номінальна | χ² та коефіцієнт Крамера (V) |
| відношень × бінарна номінальна | точковий бісеріальний коефіцієнт |
| номінальна × порядкова | коефіцієнт Крамера (V) |

Обґрунтуйте вибір, обчисліть міру й поясніть результат. Окремо покажіть, **що
станеться**, якщо застосувати Пірсона там, де він недопустимий, — і чому отримане
число не можна інтерпретувати.

In [11]:
A, B = В['пара']
print('Ваша пара:', A, '×', B)
print()
print(f'{A}: шкала —', ПАСПОРТ.get(A, ('не у вашому списку',))[0] or '(визначте самостійно)')
print(f'{B}: шкала —', ПАСПОРТ.get(B, ('не у вашому списку',))[0] or '(визначте самостійно)')
print()
print(df[[A, B]].head(6).to_string())

Ваша пара: Стать × помер/жив

Стать: шкала — не у вашому списку
помер/жив: шкала — номінальна

  Стать  помер/жив
0     ч          1
1     ч          2
2     ж          2
3     ж          2
4     ж          2
5     ж          1


In [12]:
ОБРАНА_МІРА = "Коефіцієнт V Крамера (Cramér's V) та хі-квадрат (χ²)"
ЧОМУ_САМЕ_ВОНА = 'Обидві змінні (Стать та помер/жив) належать до номінальної шкали вимірювання, тому для оцінки зв\'язку між ними використовується критерій хі-квадрат та коефіцієнт V Крамера.'

assert ОБРАНА_МІРА and len(ЧОМУ_САМЕ_ВОНА) >= 40, 'Заповніть обидва поля'

from scipy.stats import chi2_contingency
import numpy as np

df['Стать_чиста'] = df['Стать'].astype(str).str.strip().str[0].map({'ч': 'ч', 'ж': 'ж'})

contingency = pd.crosstab(df['Стать_чиста'], df[B])
print("Таблиця спряженості (Cross-tabulation):")
display(contingency)
print()

chi2, p, dof, expected = chi2_contingency(contingency, correction=False)
n = contingency.sum().sum()
min_dim = min(contingency.shape) - 1
cramers_v = np.sqrt(chi2 / (n * min_dim)) if min_dim > 0 else 0

print(f"Хі-квадрат (χ²): {chi2:.4f}")
print(f"p-значення (p-value): {p:.4f}")
print(f"Коефіцієнт V Крамера: {cramers_v:.4f}")

Таблиця спряженості (Cross-tabulation):


помер/жив,1,2
Стать_чиста,,
ж,43,32
ч,18,28



Хі-квадрат (χ²): 3.7792
p-значення (p-value): 0.0519
Коефіцієнт V Крамера: 0.1767


In [13]:
df_pearson = df[['Стать_чиста', B]].dropna().copy()
df_pearson['Стать_чиста'] = pd.factorize(df_pearson['Стать_чиста'])[0]
df_pearson[B] = pd.factorize(df_pearson[B])[0]

r_pearson = df_pearson['Стать_чиста'].corr(df_pearson[B], method='pearson')

print(f"Розрахований коефіцієнт кореляції Пірсона: {r_pearson:.4f}\n")

r_swapped = (1 - df_pearson['Стать_чиста']).corr(df_pearson[B], method='pearson')
print(f"Той самий розрахунок після зміни кодування статі (0↔1): {r_swapped:.4f}\n")

print("Чому отримане число не має сенсу і не підлягає інтерпретації:")
print("1. Кореляція Пірсона вимагає щонайменше інтервальної шкали, де числа мають фізичний зміст і між ними визначено точні відстані.")
print("2. Для номінальних змінних 'Стать' та 'помер/жив' числа (0, 1) є лише умовними ярликами, а не кількісними вимірами.")
print("3. Як видно вище, зміна кодування змінює знак r на протилежний, що доводить довільність отриманого числа.")

Розрахований коефіцієнт кореляції Пірсона: -0.1767

Той самий розрахунок після зміни кодування статі (0↔1): 0.1767

Чому отримане число не має сенсу і не підлягає інтерпретації:
1. Кореляція Пірсона вимагає щонайменше інтервальної шкали, де числа мають фізичний зміст і між ними визначено точні відстані.
2. Для номінальних змінних 'Стать' та 'помер/жив' числа (0, 1) є лише умовними ярликами, а не кількісними вимірами.
3. Як видно вище, зміна кодування змінює знак r на протилежний, що доводить довільність отриманого числа.


---
# Етап 5. Підготовка змінних до моделі

Шкала визначає не лише статистику, а й спосіб кодування.

| Шкала | Як готують до моделі |
|---|---|
| Номінальна | one-hot кодування; порядок категорій не вводять |
| Порядкова | впорядкована категорія або цілі коди із **заданим** порядком |
| Інтервальна | стандартизація; для дат — перехід до тривалостей |
| Відношень | нормалізація або стандартизація; можливе логарифмування |
| Ідентифікатор | у модель не беруть узагалі |

Для кожного свого стовпця вкажіть спосіб підготовки — і **виконайте його**
щонайменше для трьох стовпців різних шкал.

In [14]:
ПІДГОТОВКА = {
    'ПІП': 'Вилучаємо з моделі, оскільки це унікальний ідентифікатор пацієнта без математичного змісту.',
    'помер/жив': 'One-hot кодування або бінарне перетворення (0/1) без введення штучного порядку.',
    'ступінь КТ': 'Кодування цілими числами із збереженням природного порядку (легкий=1, середній=2, важкий=3, критичний=4).',
    'Дата розтину': 'Перетворення дати на інтервальну величину (кількість днів від початкової дати) з подальшою стандартизацією.',
    'КТ': 'Очищення текстових значень від відсотків, зведення до числового вигляду та min-max нормалізація.',
    'Т вище 37,5': 'Витяг числової кількості днів підвищеної температури та нормалізація шкали.',
    'Сечов1': 'Min-max нормалізація або Z-стандартизація числового лабораторного показника.',
    'БРпр1': 'Стандартизація числової концентрації за формулою (x - mean) / std.',
}

pd.DataFrame([(c, ПАСПОРТ[c][0], ПІДГОТОВКА[c]) for c in МОЇ],
             columns=['Стовпець', 'Шкала', 'Як готуємо'])

,Стовпець,Шкала,Як готуємо
0,ПІП,ідентифікатор,"Вилучаємо з моделі, оскільки це унікальний ідентифікатор..."
1,помер/жив,номінальна,One-hot кодування або бінарне перетворення (0/1) без вве...
2,ступінь КТ,порядкова,Кодування цілими числами із збереженням природного поряд...
3,Дата розтину,інтервальна,Перетворення дати на інтервальну величину (кількість дні...
4,КТ,відношень,"Очищення текстових значень від відсотків, зведення до чи..."
5,"Т вище 37,5",відношень,Витяг числової кількості днів підвищеної температури та ...
6,Сечов1,відношень,Min-max нормалізація або Z-стандартизація числового лабо...
7,БРпр1,відношень,Стандартизація числової концентрації за формулою (x - me...


In [15]:
df_prepared = pd.DataFrame(index=df.index)

df_prepared = pd.concat([df_prepared, pd.get_dummies(df['помер/жив'], prefix='стан')], axis=1)

kt_mapping = {
    'легкий': 1,
    'середній': 2,
    'сердній': 2,
    'важкий': 3,
    'критичний': 4
}
df_prepared['ступінь_КТ_ord'] = df['ступінь КТ'].map(kt_mapping)

sech1_clean = pd.to_numeric(df['Сечов1'], errors='coerce')
min_val = sech1_clean.min()
max_val = sech1_clean.max()

df_prepared['Сечов1_normalized'] = (sech1_clean - min_val) / (max_val - min_val)

print("Результат підготовки трьох змінних різних шкал:")
display(df_prepared[['стан_1', 'стан_2', 'ступінь_КТ_ord', 'Сечов1_normalized']].dropna().head(8))

Результат підготовки трьох змінних різних шкал:


,стан_1,стан_2,ступінь_КТ_ord,Сечов1_normalized
0,True,False,2.0,0.030817
1,False,True,2.0,0.053929
2,False,True,4.0,0.060092
3,False,True,4.0,0.098613
5,True,False,2.0,0.167951
6,False,True,3.0,0.040062
7,False,True,1.0,0.092450
9,True,False,4.0,0.106317


**Перед нормалізацією** обов'язково перевірте варіативність. Стовпець, у якому
всі значення однакові, дасть нуль у знаменнику — і мовчазний `NaN` замість помилки.

In [16]:
for c in МОЇ:
    з = pd.to_numeric(df[c], errors='coerce').dropna()
    if len(з) and з.nunique() == 1:
        print(f'УВАГА: {c} — сталий стовпець, нормалізація дасть ділення на нуль')
    elif len(з) and з.max() == з.min():
        print(f'УВАГА: {c} — max дорівнює min')
print('Перевірку варіативності виконано.')

Перевірку варіативності виконано.


In [17]:
# --- САМОПЕРЕВІРКА 2 ---
assert all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ), \
    'Для кожного стовпця вкажіть спосіб підготовки'
assert len(БЕЗГЛУЗДІ) >= 3, 'Етап 3 не завершено'
print('Самоперевірка 2 пройдена.')

Самоперевірка 2 пройдена.


---
# Етап 6. Виправлення чужого аналізу

Відкрийте файл **`ЛР3_помилковий_аналіз.ipynb`**. Це звіт, який виконується
без жодної помилки й містить десять хибних місць.

Для кожного з десяти заповніть рядок нижче:

* **розділ** — номер розділу звіту;
* **що не так** — яка шкала порушена або яке припущення хибне;
* **чому число неправильне** — що саме воно означає насправді;
* **як правильно** — коректний варіант або обґрунтована відмова від показника.

In [18]:
ВИПРАВЛЕННЯ = [
    (
        1,
        'Обчислення середнього арифметичного для номінальної шкали',
        'Змінна «помер/жив» містить числові коди (1 та 2) номінальних категорій. Середнє значення 1.5 між "помер" і "жив" не має жодного математичного чи медичного змісту.',
        'Для номінальної шкали можна рахувати лише моду та частоти (скільки пацієнтів і відсотків має кожна група).'
    ),
    (
        2,
        'Підрахунок частот на неочищених категоріальних даних (дублікати міток)',
        'Через відсутність очищення змінна містить синоніми ("ж" і "жін", "ч" і "чол") як окремі категорії, тому висновок колеги про наявність "4 статей" абсурдний.',
        'Перед аналізом частот треба звести всі синоніми до єдиного стандарту (наприклад, замінити "жін" на "ж", "чол" на "ч").'
    ),
    (
        3,
        'Усереднення порядкової шкали зі зламаним алфавітним сортуванням',
        '«Ступінь КТ» — це порядкова шкала, додавати її коди не можна. До того ж, pandas відсортував рівні за алфавітом («важкий», «вкрай важкий», «критичний», «легка»), що повністю знищило логічний порядок важкості.',
        'Потрібно вручну задати правильний порядок категорій (через pd.Categorical) і обчислювати медіану, а не середнє.'
    ),
    (
        4,
        'Конвертація тексту в числа з параметром errors="coerce" втратила дані',
        'Метод coerce перетворив усі текстові діапазони (наприклад, "20-30%") на порожні значення (NaN). Через це велика частина валідних даних просто випала, сильно спотворивши середнє.',
        'Треба написати функцію, яка очищає рядки від символу "%" і бере середнє арифметичне з меж діапазону (наприклад, (20+30)/2).'
    ),
    (
        5,
        'Обчислення статистик на даних із несумісним (змішаним) кодуванням',
        'В одному стовпці «spO2 без дот» частина значень записана як частки (0.9), а інша — як відсотки (84.0). Це катастрофічно занижує загальне середнє та мінімум.',
        'Спочатку треба привести всі числа до єдиного масштабу (наприклад, домножити всі десяткові дроби, менші за 1, на 100).'
    ),
    (
        6,
        'Обчислення коефіцієнта варіації для інтервальної шкали (дат)',
        'Дати не мають абсолютного нуля (початок відліку умовний). Тому перетворення їх на "дні від початку епохи" і ділення стандартного відхилення на середню дату дає безглузде число.',
        'Для оцінки розкиду дат достатньо подивитися на стандартне відхилення у днях (без ділення на середнє) або рахувати тривалості.'
    ),
    (
        7,
        'Кореляція Пірсона застосована до двох номінальних змінних',
        'Критерій Пірсона вимагає наявності хоча б інтервальної шкали. Для номінальних міток («Стать» та «помер/жив») отриманий коефіцієнт r=0.177 є випадковим і не підлягає інтерпретації.',
        'Зв\'язок між двома номінальними змінними треба оцінювати через таблицю сопряженості та критерій хі-квадрат (коефіцієнт Крамера).'
    ),
    (
        8,
        'Нормалізація (x-min)/(max-min) без перевірки на нульову дисперсію',
        'У стовпці «ФЕРИТ2» всі значення однакові (сталий стовпець). Тому min дорівнює max, знаменник стає нулем, і замість чисел генерується стовпець з порожніми NaN.',
        'Перед нормалізацією обов\'язково перевіряти дисперсію (чи max > min) і не нормалізувати стовпці, що є константами.'
    ),
    (
        9,
        'Наївне видалення пропусків df.dropna() на широкому датасеті',
        'Оскільки датасет має 77 ознак, майже в кожному рядку є хоча б один пропущений запис. Функція dropna() мовчки видалила 100% пацієнтів, залишивши порожній датафрейм із 0 рядків.',
        'Треба заповнювати пропуски (медіаною/модою), видаляти найбільш порожні стовпці, або використовувати алгоритми, стійкі до NaN.'
    ),
    (
        10,
        'Використання наслідку як предиктора (витік даних / Data Leakage)',
        '«Дата розтину» фізично існує тільки для померлих пацієнтів. Кореляція r=1.0 тривіальна, бо ця колонка є прямим наслідком смерті, а не фактором, який можна використати для прогнозу при госпіталізації.',
        'Стовпець «Дата розтину» потрібно обов\'язково вилучити з датасету (drop) перед побудовою будь-яких прогнозних моделей летальності.'
    )
]

assert len(ВИПРАВЛЕННЯ) == 10, f'Знайдено {len(ВИПРАВЛЕННЯ)} помилок із 10'
for р in ВИПРАВЛЕННЯ:
    assert len(р) == 4 and all(str(x).strip() for x in р), 'Заповніть усі чотири поля'
    assert len(str(р[2])) >= 30 and len(str(р[3])) >= 25, 'Пояснення закороткі'

pd.DataFrame(ВИПРАВЛЕННЯ, columns=['Розділ', 'Що не так', 'Чому число неправильне', 'Як правильно'])

,Розділ,Що не так,Чому число неправильне,Як правильно
0,1,Обчислення середнього арифметичного для номінальної шкали,Змінна «помер/жив» містить числові коди (1 та 2) номінал...,Для номінальної шкали можна рахувати лише моду та частот...
1,2,Підрахунок частот на неочищених категоріальних даних (ду...,"Через відсутність очищення змінна містить синоніми (""ж"" ...",Перед аналізом частот треба звести всі синоніми до єдино...
2,3,Усереднення порядкової шкали зі зламаним алфавітним сорт...,"«Ступінь КТ» — це порядкова шкала, додавати її коди не м...",Потрібно вручну задати правильний порядок категорій (чер...
3,4,"Конвертація тексту в числа з параметром errors=""coerce"" ...",Метод coerce перетворив усі текстові діапазони (наприкла...,"Треба написати функцію, яка очищає рядки від символу ""%""..."
4,5,Обчислення статистик на даних із несумісним (змішаним) к...,В одному стовпці «spO2 без дот» частина значень записана...,Спочатку треба привести всі числа до єдиного масштабу (н...
5,6,Обчислення коефіцієнта варіації для інтервальної шкали (...,Дати не мають абсолютного нуля (початок відліку умовний)...,Для оцінки розкиду дат достатньо подивитися на стандартн...
6,7,Кореляція Пірсона застосована до двох номінальних змінних,Критерій Пірсона вимагає наявності хоча б інтервальної ш...,Зв'язок між двома номінальними змінними треба оцінювати ...
7,8,Нормалізація (x-min)/(max-min) без перевірки на нульову ...,У стовпці «ФЕРИТ2» всі значення однакові (сталий стовпец...,Перед нормалізацією обов'язково перевіряти дисперсію (чи...
8,9,Наївне видалення пропусків df.dropna() на широкому датасеті,"Оскільки датасет має 77 ознак, майже в кожному рядку є х...","Треба заповнювати пропуски (медіаною/модою), видаляти на..."
9,10,Використання наслідку як предиктора (витік даних / Data ...,«Дата розтину» фізично існує тільки для померлих пацієнт...,Стовпець «Дата розтину» потрібно обов'язково вилучити з ...


### Виправте щонайменше три помилки в коді

Оберіть три з десяти та покажіть правильне обчислення поруч із неправильним.

In [19]:
# 1. Помилка: обчислення середнього для номінальної змінної
print("--- 1. Змінна 'помер/жив' (номінальна) ---")
print("ХИБНО (середнє арифметичне між мітками 1 і 2):")
print(round(df['помер/жив'].mean(), 2))
print("\nПРАВИЛЬНО (мода та відносні частоти у відсотках):")
print(df['помер/жив'].value_counts(normalize=True).round(4) * 100)
print("\n" + "="*50 + "\n")


# 2. Помилка: відсутність очищення текстових діапазонів перед аналізом
print("--- 2. Змінна 'КТ' (містить текст типу '20-30%', 'до 17%') ---")
print("ХИБНО: стандартний df.describe() або mean() ігнорує такі дані або видає помилку.")

def clean_kt(x):
    if pd.isna(x): return np.nan
    s = str(x).replace('%', '').replace('до ', '').strip()
    if '-' in s:
        parts = s.split('-')
        return (float(parts[0]) + float(parts[1])) / 2
    try:
        return float(s)
    except ValueError:
        return np.nan

kt_cleaned = df['КТ'].apply(clean_kt)
print("ПРАВИЛЬНО (середнє значення КТ після очищення тексту та усереднення діапазонів):")
print(round(kt_cleaned.mean(), 2))
print("\n" + "="*50 + "\n")


# 3. Помилка: подача категорій у модель без One-Hot кодування
print("--- 3. Підготовка номінальної змінної 'помер/жив' до машинного навчання ---")
print("ХИБНО (залишати як є — модель сприйме це як порядкову величину):")
print(df[['помер/жив']].head(3))
print("\nПРАВИЛЬНО (використати One-Hot кодування):")
print(pd.get_dummies(df['помер/жив'], prefix='стан', dtype=int).head(3))

--- 1. Змінна 'помер/жив' (номінальна) ---
ХИБНО (середнє арифметичне між мітками 1 і 2):
1.5

ПРАВИЛЬНО (мода та відносні частоти у відсотках):
помер/жив
1    50.41
2    49.59
Name: proportion, dtype: float64


--- 2. Змінна 'КТ' (містить текст типу '20-30%', 'до 17%') ---
ХИБНО: стандартний df.describe() або mean() ігнорує такі дані або видає помилку.
ПРАВИЛЬНО (середнє значення КТ після очищення тексту та усереднення діапазонів):
17.91


--- 3. Підготовка номінальної змінної 'помер/жив' до машинного навчання ---
ХИБНО (залишати як є — модель сприйме це як порядкову величину):
   помер/жив
0          1
1          2
2          2

ПРАВИЛЬНО (використати One-Hot кодування):
   стан_1  стан_2
0       1       0
1       0       1
2       0       1


---
# Етап 7. Висновки та індивідуальне питання

In [20]:
ВИСНОВКИ = [
    "Аналіз шкал вимірювання показав, що наші медичні дані містять змінні різних типів (від номінальної 'помер/жив' до шкали відношень 'Сечов1'), що категорично забороняє використовувати однакові метрики для всіх колонок.",
    "Після очищення 'Стать' до двох категорій (ж/ч) критерій хі-квадрат і коефіцієнт V Крамера (≈0.18, p≈0.05) показали слабкий зв'язок статі з результатом лікування, який не досягає значущості на рівні 0.05; без очищення (4 «статі») V хибно завищується до 0.30.",
    "Дослідження датасету довело, що сирі змінні (наприклад, 'КТ' із текстовими діапазонами, 'Стать' із синонімами) потребують обов'язкового очищення, а категоріальні дані — One-Hot кодування перед подачею в алгоритми машинного навчання.",
]

assert len(ВИСНОВКИ) >= 3 and all(len(в) >= 60 for в in ВИСНОВКИ), \
    'Потрібні три змістовні висновки'
for i, в in enumerate(ВИСНОВКИ, 1):
    print(f'{i}. {в}')

1. Аналіз шкал вимірювання показав, що наші медичні дані містять змінні різних типів (від номінальної 'помер/жив' до шкали відношень 'Сечов1'), що категорично забороняє використовувати однакові метрики для всіх колонок.
2. Після очищення 'Стать' до двох категорій (ж/ч) критерій хі-квадрат і коефіцієнт V Крамера (≈0.18, p≈0.05) показали слабкий зв'язок статі з результатом лікування, який не досягає значущості на рівні 0.05; без очищення (4 «статі») V хибно завищується до 0.30.
3. Дослідження датасету довело, що сирі змінні (наприклад, 'КТ' із текстовими діапазонами, 'Стать' із синонімами) потребують обов'язкового очищення, а категоріальні дані — One-Hot кодування перед подачею в алгоритми машинного навчання.


In [21]:
print('ВАШЕ ПИТАННЯ:'); print(В['питання']); print()

ВІДПОВІДЬ = '''Формула (x − min) / (max − min) зламається на стовпці «ФЕРИТ2»: у ньому лише одне непорожнє значення (1265), тобто стовпець фактично є константою.
Для нього min = max = 1265, тому знаменник (max − min) дорівнює нулю, а чисельник (x − min) теж нуль або NaN. Ділення нуля на нуль дає NaN, тому весь стовпець після нормалізації складається з NaN (у звіті колеги це видно: count = 0, mean = NaN).
Python не зупиняє програму помилкою, а мовчки повертає NaN, тож модель впаде пізніше й в іншому місці. Це видно навіть на основі лише 1 непорожнього значення з 121 рядка: стовпець не несе жодної інформації.
Тому перед min-max нормалізацією треба перевіряти варіативність (nunique() > 1 або max > min) і виключати сталі стовпці з обробки.'''

assert len(ВІДПОВІДЬ.strip()) >= 200, 'Відповідь має бути розгорнутою (від 200 символів)'
print(ВІДПОВІДЬ)

ВАШЕ ПИТАННЯ:
Ви хочете нормалізувати всі числові стовпці формулою (x − min) / (max − min). Знайдіть у файлі стовпець, на якому ця формула зламається, і поясніть чому.

Формула (x − min) / (max − min) зламається на стовпці «ФЕРИТ2»: у ньому лише одне непорожнє значення (1265), тобто стовпець фактично є константою.
Для нього min = max = 1265, тому знаменник (max − min) дорівнює нулю, а чисельник (x − min) теж нуль або NaN. Ділення нуля на нуль дає NaN, тому весь стовпець після нормалізації складається з NaN (у звіті колеги це видно: count = 0, mean = NaN).
Python не зупиняє програму помилкою, а мовчки повертає NaN, тож модель впаде пізніше й в іншому місці. Це видно навіть на основі лише 1 непорожнього значення з 121 рядка: стовпець не несе жодної інформації.
Тому перед min-max нормалізацією треба перевіряти варіативність (nunique() > 1 або max > min) і виключати сталі стовпці з обробки.


---
# Етап 8. Декларація про використання інструментів ШІ

Користуватися мовними моделями дозволено. Це нормальна інженерна практика.

**Обов'язковою є прозорість:** ви вказуєте, де саме і як скористалися інструментом —
так само, як у науковій статті вказують використане програмне забезпечення.
Задекларована допомога порушенням не є.

Порушенням є інше: здати роботу, яку ви не можете пояснити. На захисті вас
попросять показати конкретний рядок свого коду, пояснити, чому він саме такий,
і внести в нього невелику зміну на місці.

Для кожного етапу оберіть одне з формулювань:

* `'ні'` — інструментами ШІ не користувався
* `'синтаксис'` — питав про назву функції або синтаксис, код писав сам
* `'код, перевірено'` — код згенеровано, я його прочитав, перевірив і розумію
* `'текст, перероблено'` — згенеровано чернетку тексту, яку я переписав по суті
* `'текст, як є'` — згенерований текст залишено майже без змін

In [22]:
ДЕКЛАРАЦІЯ = {
    'етап 1 — паспорт змінних':        'код, перевірено',
    'етап 2 — допустимі статистики':   'код, перевірено',
    'етап 3 — безглузді числа':        'код, перевірено',
    'етап 4 — міра зв’язку':           'код, перевірено',
    'етап 5 — підготовка до моделі':   'код, перевірено',
    'етап 6 — виправлення аналізу':    'текст, як є',
    'етап 7 — висновки й відповідь':   'текст, як є',
}

ЩО_САМЕ_Я_ПЕРЕВІРИВ = 'Для згенерованого коду я детально перевірив логіку обчислення статистик залежно від шкал вимірювання, правильність використання критерію хі-квадрат та коефіцієнта Крамера для номінальних даних, а також коректність нормалізації та One-Hot кодування.'

ПІДТВЕРДЖУЮ = True     # я можу пояснити кожен рядок свого коду
                        # і внести в нього зміни на вимогу під час захисту

In [23]:
# --- ФІНАЛЬНА САМОПЕРЕВІРКА ---
ФОРМУЛЮВАННЯ = {'ні', 'синтаксис', 'код, перевірено', 'текст, перероблено', 'текст, як є'}

перевірки = {
    'заповнено титульні поля':        bool(ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР),
    'етап 1: паспорт із 8 стовпців':  all(ПАСПОРТ[c][0] and ПАСПОРТ[c][1] for c in МОЇ),
    'етап 3: три безглузді числа':    len(БЕЗГЛУЗДІ) >= 3,
    'етап 4: міру обрано й пояснено': bool(ОБРАНА_МІРА and len(ЧОМУ_САМЕ_ВОНА) >= 40),
    'етап 5: підготовку описано':     all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ),
    'етап 6: знайдено 10 помилок':    len(ВИПРАВЛЕННЯ) == 10,
    'етап 7: три висновки':           len(ВИСНОВКИ) >= 3,
    'етап 7: відповідь на питання':   len(ВІДПОВІДЬ.strip()) >= 200,
    'декларацію заповнено':           all(v in ФОРМУЛЮВАННЯ for v in ДЕКЛАРАЦІЯ.values()),
    'опис перевірки ≥ 120 символів':  len(ЩО_САМЕ_Я_ПЕРЕВІРИВ.strip()) >= 120,
    'підтвердження авторства':        ПІДТВЕРДЖУЮ is True,
}

print('ЧЕК-ЛИСТ ГОТОВНОСТІ')
print('=' * 52)
for назва, ок in перевірки.items():
    print(('  ГОТОВО     ' if ок else '  НЕ ГОТОВО ') + назва)
print('=' * 52)

assert all(перевірки.values()), 'Робота не готова до здачі — див. чек-лист вище'
print()
print(f'Робота варіанта {В["варіант"]} готова до захисту.')
print(f'{ПРІЗВИЩЕ_ІМЯ}, {ГРУПА}')

ЧЕК-ЛИСТ ГОТОВНОСТІ
  ГОТОВО     заповнено титульні поля
  ГОТОВО     етап 1: паспорт із 8 стовпців
  ГОТОВО     етап 3: три безглузді числа
  ГОТОВО     етап 4: міру обрано й пояснено
  ГОТОВО     етап 5: підготовку описано
  ГОТОВО     етап 6: знайдено 10 помилок
  ГОТОВО     етап 7: три висновки
  ГОТОВО     етап 7: відповідь на питання
  ГОТОВО     декларацію заповнено
  ГОТОВО     опис перевірки ≥ 120 символів
  ГОТОВО     підтвердження авторства

Робота варіанта 7 готова до захисту.
Токарик Сергій, ШІ-33


---
### Перед здачею

Виконайте **Kernel → Restart & Run All**. Ноутбук має пройти згори донизу
без жодної помилки — це перше, що перевіряється на захисті.